# 嗅觉受体分子对接探索记录

先运行下面的路径配置单元格，并阅读 README 的运行说明。
本文件保留探索过程及中间检查步骤；后续单元格会访问网络、下载数据、转换结构或运行对接。
配体准备、质量分级和结合能汇总仍包含手动步骤。发布前已清空输出；请按阶段运行。


In [ ]:
from pathlib import Path
import os

# 从项目根目录启动 Jupyter；所有数据路径均相对于该目录。
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "01_fetch_OR_genes.ipynb").is_file():
    raise RuntimeError("请从项目根目录启动 Jupyter 后再运行此 Notebook。")
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
VINA_PATH = os.environ.get("VINA_PATH", "./bin/vina.exe")
OBABEL_PATH = os.environ.get("OBABEL_PATH", "./bin/OpenBabel-3.1.1/obabel.exe")
for directory in [DATA_DIR, FIGURES_DIR, DATA_DIR / "receptors",
                  DATA_DIR / "ligands", DATA_DIR / "docked"]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import requests

print("pandas 版本", pd.__version__)
print("requests 已导入")

In [ ]:
import pandas as pd
import requests
import io

url="https://rest.ensembl.org/xrefs/symbol/homo_sapiens?prefix=OR"

response = requests.get(url, headers={"Content-Type":"application/json"})

if response.status_code ==200:
    data = response.json()
    print(f"successfully get {len(data)}条数据")
    print(data[:3])
else:
    print(f"fail to get response,status code:{response.status_code}")
    data=[]

In [ ]:
import pandas as pd

df=pd.read_csv("./mart_export.txt",low_memory=False)

print(f"总基因数：{len(df)}")
print(f"列名:{df.columns.tolist()}")
print(df.head())

In [ ]:
or_all = df[df["Gene name"].str.startswith("OR",na=False)]
print(f"包含 OR 的所有行：{len(or_all)}")

or_unique=or_all.drop_duplicates(subset=["Gene name"])
print(f"去重后唯一的 OR 基因数：{len(or_unique)}")

print(or_unique["Gene name"].head(10))

In [ ]:
or_unique.to_csv("./data/OR_genes_unique.csv",index=False)

gene_names=or_unique["Gene name"].to_frame()
gene_names.to_csv("./data/OR_gene_names.csv",index=False)

print("已保存两个文件")
print(" -or_genes_unique.csv(完整信息)")
print(" -or_genes_names.csv(仅基因名)")

In [ ]:
import pandas as pd

gene_df=pd.read_csv("./data/OR_gene_names.csv")
gene_list= gene_df["Gene name"].tolist()

print(f"共加载{len(gene_list)}个基因名")
print("前10个：",gene_list[:10])

In [ ]:
import requests
import time

gene_list=gene_df["Gene name"].tolist()

def fetch_uniport_sequences(gene_names,batch_size=100):
    all_seq=[]

    for i in range(0,len(gene_names),batch_size):
        batch=gene_names[i:i+batch_size]
        #构造查询字符串
        query=" OR ".join([f"gene:{g}" for g in batch])
        #加上物种限定
        query=f"({query}) AND organism_id:9606"
        
        url= "https://rest.uniprot.org/uniprotkb/stream"
        params={
            "format":"fasta",
            "query":query
        }

        print(f"正在获取第{i//batch_size +1} 批 ({len(batch)}个基因)...")
        response =requests.get(url,params=params)

        if response.status_code == 200:
            sequences = response.text
            all_seq.append(sequences)
            print(f" √ 成功获取{len(batch)}个序列😋")
        else:
            print(f" × 第{i//batch_size+1}批失败😭，状态码：{response.status_code}")
            print(f" 响应内容：{response.text[:200]}")

        time.sleep(0.5)

    return "\n".join(all_seq)

#执行获取
fasta_data=fetch_uniport_sequences(gene_list)

#保存FASTA文件
output_path="./data/OR_proteins.fasta"
with open(output_path,"w") as f:
    f.write(fasta_data)
print(f"\n √ 已保存FASTA文件到：{output_path},完美收工😎👍")

In [ ]:
import re

fasta_file="./data/OR_proteins.fasta"
non_or_genes=[]

with open(fasta_file,"r") as f:
    for line in f:
        if line.startswith(">"):
            #假基因名称
            match=re.search(r"GN=(\S+)",line)
            if match:
                gene=match.group(1)
                #非OR开头就记录
            if not gene.startswith("OR"):
                non_or_genes.append(gene)

unique_non_or=set(non_or_genes)
print(f"发现{len(unique_non_or)}个非 OR 开头的基因：")
for g in sorted(unique_non_or):
    print(f"{g}")

In [ ]:
import pandas as pd

# 重新读取原始数据（所有人类基因）
df = pd.read_csv("./mart_export.txt", low_memory=False)

# 精确筛选：以 OR 开头，且第三个字符是数字（标准的嗅觉受体命名）
# 嗅觉受体基因名格式：OR + 数字 + 字母/数字（如 OR1A1, OR51E2）
or_pattern = r"^OR\d"  # 以 OR 开头，后面紧跟数字
or_genes = df[df["Gene name"].str.match(or_pattern, na=False)]

# 去重
or_unique_clean = or_genes.drop_duplicates(subset=["Gene name"])

# 保存
or_unique_clean.to_csv("./data/or_genes_clean.csv", index=False)

print(f"修正后 OR 基因数：{len(or_unique_clean)}")
print(or_unique_clean["Gene name"].head(10))

In [ ]:
import requests
import time
import pandas as pd

clean_df=pd.read_csv("./data/or_genes_clean.csv")
gene_list=clean_df["Gene name"].tolist()

print(f"准备获取{len(gene_list)}个基因的蛋白质序列")

def fetch_uniport_sequences(gene_names,batch_size=100):
    all_seq=[]
    for i in range(0,len(gene_names),batch_size):
        batch=gene_names[i:i+batch_size]
        query=" OR ".join([f"gene:{g}" for g in batch])
        query=f"({query}) AND organism_id:9606"

        url="https://rest.uniprot.org/uniprotkb/stream"
        params={"format":"fasta","query":query}

        print(f"正在获取第{i//batch_size+1}批({len(batch)}个基因)")
        response=requests.get(url,params=params)

        if response.status_code ==200:
            all_seq.append(response.text)
            print(f" ✅成功获取第 {len(batch)}  批🤓👌")
        else:
            print(f" ❌失败了喵！状态码：{response.status_code}🖐️😭🖐️")

        time.sleep(0.5)

    return "\n".join(all_seq)

fasta_data=fetch_uniport_sequences(gene_list)

output_path="./data/OR_proteins_clean.fasta"
with open(output_path,"w") as f:
    f.write(fasta_data)

print(f"\n √ 已保存干净的FASTA文件到：{output_path}😋👍")

In [ ]:
import re

fasta_file = "./data/OR_proteins_clean.fasta"

with open(fasta_file, "r") as f:
    for line in f:
        if line.startswith(">"):
            # 提取 GN= 后面的部分（直到空格或分号）
            match = re.search(r"GN=([^\s;]+)", line)
            if match:
                gene = match.group(1)
                # 如果 GN 里没有 "OR" 字样，打印出来供人工确认
                if "OR" not in gene:
                    print(f"警告：{gene} 不是 OR 前缀，完整标题行：\n{line.strip()}")
print("没毛病嗷😎👍")

In [ ]:
import requests
import pandas as pd

gene_df=pd.read_csv("./data/or_genes_clean.csv")
gene_list=gene_df["Gene name"].tolist()

def check_alphafold(gene):
    url=f"https://alphafold.ebi.ac.uk/api/prediction/{gene}"
    try:
        response=requests.get(url,timeout=10)
        if response.status_code == 200:
            data=response.json()
            if data:
                plddt=data[0].get("plddt",[])
                avg=sum(plddt)/len(plddt) if plddt else 0
                return avg
    except:
        pass
    return None

#扫描前30个找置信度最高
results=[]
for g in gene_list[:30]:
    avg=check_alphafold(g)
    if avg and avg > 70:
        results.append((g,avg))
        print(f"{g}平均置信度{avg:.1f}")
if results:
    best=max(results,key=lambda x:x[1])
    print(f"\n√ 推荐靶点：{best[0]}(置信度{best[1]:.1f})")
else:
    print("前30个未找到高度信度结构，扩大扫描")

In [ ]:
import pandas as pd

df_check=pd.read_csv("./data/or_genes_clean.csv",encoding='utf-8')

print("列名列表: ")
for i, col in enumerate(df_check.columns):
    print(f"{i}:'{col}'(长度:{len(col)})")

print("\n前三行数据：")
print(df_check.head(3))

In [ ]:
import pandas as pd

df=pd.read_csv("./data/or_genes_clean.csv")

gene_list=df[["Gene name","UniProtKB/Swiss-Prot ID"]].copy()

#过滤出有UinPort ID的基因
has_uniprot=gene_list.dropna(subset=["UniProtKB/Swiss-Prot ID"]).copy()
missing_uniprot=gene_list[gene_list["UniProtKB/Swiss-Prot ID"].isna()]

print(f"共有{len(gene_list)}个基因")
print(f"有 UniProt ID的：{len(has_uniprot)}个")
print(f"缺 UniProt ID的：{len(missing_uniprot)}个")

has_uniprot["AlphaFold_URL"]="https://alphafold.ebi.ac.uk/files/" + has_uniprot["UniProtKB/Swiss-Prot ID"] + "-F1-model_v4.pdb"

print("\n前5个可用的AlphaFold 结构链接：")
print(has_uniprot[["Gene name","UniProtKB/Swiss-Prot ID","AlphaFold_URL"]].head())

In [ ]:
# 查一下这两个目标基因的 AlphaFold 下载链接
targets = ["OR1A1", "OR51E2"]
has_uniprot[has_uniprot["Gene name"].isin(targets)][["Gene name", "UniProtKB/Swiss-Prot ID", "AlphaFold_URL"]]

In [ ]:
# 提取 OR1A1 和 OR51E2 的完整链接
target_links = has_uniprot[has_uniprot["Gene name"].isin(["OR1A1", "OR51E2"])][["Gene name", "AlphaFold_URL"]]

for idx, row in target_links.iterrows():
    print(f"{row['Gene name']} 的完整下载链接：")
    print(row['AlphaFold_URL'])
    print("-" * 50)

In [ ]:
import requests

def get_alphafold_download_url(uniprot_id):
    """
    通过 AlphaFold API 查询给定 UniProt ID 的实际 PDB 下载链接。
    """
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    response = requests.get(api_url)
    
    if response.status_code != 200:
        return f"API 请求失败，状态码：{response.status_code}"
    
    data = response.json()
    if not data:
        return "未找到该 UniProt ID 对应的结构。"
    
    # 从返回的 JSON 中提取下载链接
    pdb_url = data[0].get("pdbUrl")
    if pdb_url:
        return pdb_url
    else:
        return "该条目没有提供 PDB 下载链接。"

# 测试 OR1A1
uniprot_id = "Q9P1Q5"
url = get_alphafold_download_url(uniprot_id)
print(f"UniProt ID {uniprot_id} 的 AlphaFold 下载链接：")
print(url)

In [ ]:
import re
def clean_fasta_header(input_file,output_file):


    with open(input_file,'r') as infile,open(output_file,'w') as outfile:
        for line in infile:
            if line.startswith('>'):
                match=re.search(r'GN=([A-Z0-9]+)',line)
                if match:
                    new_header=f">{match.group(1)}\n"
                else:
                    new_header=line

                outfile.write(new_header)
            else:
                outfile.write(line)

input_path="./data/OR_proteins_clean.fasta"
output_path="./data/or_proteins_clean_names.fasta"

clean_fasta_header(input_path,output_path)
print(f"清洗完成！新文件已保存为：{output_path}")

In [ ]:
import re
from collections import Counter

def clean_fasta_header_unique(input_file, output_file):
    """
    读取 FASTA 文件，提取 GN= 后面的基因名，并附加 UniProt ID 的一部分以保证唯一性。
    如果找不到 GN=，则保留原始头信息。
    """
    # 第一步：读取所有序列，统计 GN 名称出现的次数
    headers = []
    sequences = []
    gn_list = []
    with open(input_file, 'r') as infile:
        lines = infile.readlines()
        i = 0
        while i < len(lines):
            if lines[i].startswith('>'):
                headers.append(lines[i].strip())
                # 提取 GN 和 UniProt ID
                gn_match = re.search(r'GN=([A-Z0-9]+)', lines[i])
                sp_match = re.search(r'sp\|([A-Z0-9]+)\|', lines[i])
                gn = gn_match.group(1) if gn_match else "Unknown"
                sp = sp_match.group(1) if sp_match else "unknown"
                gn_list.append((gn, sp))
                i += 1
                # 读取序列行（可能有多行，直到遇到下一个 > 或文件结束）
                seq_lines = []
                while i < len(lines) and not lines[i].startswith('>'):
                    seq_lines.append(lines[i].strip())
                    i += 1
                sequences.append(''.join(seq_lines))
            else:
                i += 1

    # 第二步：统计每个 GN 名称的出现次数，用于添加唯一后缀
    gn_counts = Counter([item[0] for item in gn_list])

    # 第三步：生成新的文件，为重复的 GN 添加后缀
    with open(output_file, 'w') as outfile:
        for idx, (header, seq) in enumerate(zip(headers, sequences)):
            gn, sp = gn_list[idx]
            if gn_counts[gn] > 1:
                # 如果该 GN 有重复，在名称后加上 UniProt ID 以确保唯一
                new_header = f">{gn}_{sp}"
            else:
                # 如果该 GN 唯一，直接使用 GN 名称
                new_header = f">{gn}"
            outfile.write(new_header + '\n')
            outfile.write(seq + '\n')

    print(f"清洗完成！新文件已保存为: {output_file}")
    print(f"总共处理了 {len(headers)} 条序列，发现 {sum(1 for c in gn_counts.values() if c > 1)} 个有重复的基因家族。")

# 请替换为你的实际文件路径
input_path = "./data/OR_proteins_clean.fasta"
output_path = "./data/or_proteins_clean_names_unique.fasta"

clean_fasta_header_unique(input_path, output_path)

In [ ]:
import pandas as pd

# 重新读取你已有的 OR 表
df = pd.read_csv("./data/or_genes_clean.csv")

# 拆出有 UniProt ID 的行
has_uniprot = df[df["UniProtKB/Swiss-Prot ID"].notna()].copy()

# 构造 AlphaFold 下载链接
has_uniprot["AlphaFold_URL"] = "https://alphafold.ebi.ac.uk/files/AF-" + has_uniprot["UniProtKB/Swiss-Prot ID"] + "-F1-model_v4.pdb"

print(f"共 {len(has_uniprot)} 个受体有 AlphaFold 结构可用")

In [ ]:
import pandas as pd
import requests
import time
import os

# 读取表格
df = pd.read_csv("./data/or_genes_clean.csv")
has_uniprot = df[df["UniProtKB/Swiss-Prot ID"].notna()].copy()

# 目标受体列表（使用 UniProt ID，而不是基因名）
target_uniprot_ids = [
    "Q9P1Q5",  # OR1A1
    "P34982",  # OR1D2
    "Q6IEY1",  # OR4F3
    "P30954",  # OR10J1
    "Q8TCB6",  # OR51E1
    "Q8NGL4",  # OR5A1
    "Q6IF80",  # OR2A1
    "Q13606",  # OR2F1
    "Q8NG98",  # OR7D4
    "Q8NGJ5",  # OR52E1
]

def get_alphafold_pdb_url(uniprot_id):
    """通过 AlphaFold API 获取真实的 PDB 下载链接"""
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    try:
        resp = requests.get(api_url, timeout=15)
        if resp.status_code == 200:
            data = resp.json()
            if data and isinstance(data, list) and len(data) > 0:
                pdb_url = data[0].get("pdbUrl")
                if pdb_url:
                    return pdb_url
    except:
        pass
    return None

# 创建保存目录
os.makedirs("./data/receptors", exist_ok=True)

# 逐个下载
for uniprot_id in target_uniprot_ids:
    # 获取对应的基因名
    gene_row = has_uniprot[has_uniprot["UniProtKB/Swiss-Prot ID"] == uniprot_id]
    gene_name = gene_row["Gene name"].values[0] if not gene_row.empty else uniprot_id
    
    url = get_alphafold_pdb_url(uniprot_id)
    if url:
        response = requests.get(url)
        if response.status_code == 200:
            filepath = f"./data/receptors/{gene_name}.pdb"
            with open(filepath, "w") as f:
                f.write(response.text)
            print(f"✅ {gene_name} ({uniprot_id}) 下载成功")
        else:
            print(f"❌ {gene_name} ({uniprot_id}) PDB 下载失败，状态码：{response.status_code}")
    else:
        print(f"⚠️ {gene_name} ({uniprot_id}) 未在 AlphaFold 中找到结构")
    
    time.sleep(1)

print(f"全部下载完成！！！😎👌")

In [ ]:
import os
import pandas as pd

# 读取基因名 ↔ Uniprot ID 的映射表
df = pd.read_csv("./data/or_genes_clean.csv")
id_to_gene = dict(zip(df["UniProtKB/Swiss-Prot ID"], df["Gene name"]))

# 列出 receptors 目录下的所有 .pdb 文件
pdb_files = os.listdir("./data/receptors/")
pdb_files = [f for f in pdb_files if f.endswith(".pdb")]

print("当前 receptors/ 目录下的 PDB 文件：")
for f in pdb_files:
    print(f"  {f}")

# 重命名为基因名
rename_log = []
for f in pdb_files:
    # 提取文件名中的 Uniprot ID（去掉扩展名）
    file_id = f.replace(".pdb", "")
    if file_id in id_to_gene:
        new_name = f"{id_to_gene[file_id]}.pdb"
        if f != new_name:
            old_path = f"./data/receptors/{f}"
            new_path = f"./data/receptors/{new_name}"
            os.rename(old_path, new_path)
            rename_log.append(f"{f} → {new_name}")
    elif file_id in id_to_gene.values():
        # 已经是基因名，跳过
        continue
    else:
        print(f"⚠️ 未找到 {file_id} 对应的基因名，请手动检查")

print("\n重命名记录：")
for log in rename_log:
    print(f"  {log}")

In [ ]:
import os

obabel = OBABEL_PATH
receptor_dir = r"./data/receptors"

pdb_files = [f for f in os.listdir(receptor_dir) if f.endswith(".pdb") and "_clean" not in f]

for pdb in pdb_files:
    name = pdb.replace(".pdb", "")
    inp = os.path.join(receptor_dir, pdb)
    out = os.path.join(receptor_dir, f"{name}_fixed.pdbqt")
    cmd = f'"{obabel}" "{inp}" -O "{out}" -xr -h'
    print(f"转换: {pdb}")
    !{cmd}
    print(f"完成: {name}_fixed.pdbqt\n")

In [ ]:
#文件检查
import os
receptor_dir = r"./data/receptors"
pdbqt_files = [f for f in os.listdir(receptor_dir) if f.endswith("_fixed.pdbqt")]
print(f"已生成 {len(pdbqt_files)} 个 PDBQT 文件")
print(pdbqt_files)

In [ ]:
import os
import subprocess

vina_path = VINA_PATH
receptor_dir = r"./data/receptors"
ligand_dir = r"./data/ligands"
output_dir = r"./data/docked"

# ===== 坐标表（手动录入） =====
center_coords = {
    "OR5I1":   (8.2455, -4.9405, 4.7851),
    "OR7D4":   (1.9277, -5.6195, 1.5193),
    "OR5D13":  (1.2571, 1.5287, -5.9231),
    "OR51L1":  (1.9054, 1.5593, 1.2470),
    "OR4F3":   (3.5756, 2.8390, 0.6677),
    "OR51E1":  (1.8568, -2.0996, -4.2105),
    "OR1D2":   (5.2149, 1.6629, 0.9106),
    "OR10J1":  (-7.1647, 12.367, -2.1007),
}

size_x = size_y = size_z = 25
exhaustiveness = 8
num_modes = 9
cpu = 8

# 配体列表（此处换成你实际想跑的配体文件，不含扩展名）
ligand_list = ["citronellal", "eugenol", "limonene", "linalool"]

# 扫描已有受体
receptor_files = [f for f in os.listdir(receptor_dir) if f.endswith("_fixed.pdbqt")]

os.makedirs(output_dir, exist_ok=True)

for receptor_file in receptor_files:
    receptor_name = receptor_file.replace("_fixed.pdbqt", "")
    if receptor_name not in center_coords:
        print(f"⚠️ 跳过 {receptor_name}：未在坐标表中找到")
        continue

    cx, cy, cz = center_coords[receptor_name]
    receptor_path = os.path.join(receptor_dir, receptor_file)

    for ligand_name in ligand_list:
        ligand_path = os.path.join(ligand_dir, f"{ligand_name}.pdbqt")
        if not os.path.exists(ligand_path):
            print(f"⚠️ 跳过 {receptor_name}+{ligand_name}：配体文件不存在")
            continue

        output_file = f"{receptor_name}_{ligand_name}.pdbqt"
        output_path = os.path.join(output_dir, output_file)

        cmd = (
            f'"{vina_path}"'
            f' --receptor "{receptor_path}"'
            f' --ligand "{ligand_path}"'
            f' --center_x {cx} --center_y {cy} --center_z {cz}'
            f' --size_x {size_x} --size_y {size_y} --size_z {size_z}'
            f' --exhaustiveness {exhaustiveness} --num_modes {num_modes} --cpu {cpu}'
            f' --out "{output_path}"'
        )

        print(f"对接: {receptor_name} + {ligand_name}")
        subprocess.run(cmd, shell=True)

In [ ]:
import os
ligand_dir=r"./data/ligands"
print("当前ligands/ 目录下的文件：")
for f in os.listdir(ligand_dir):
    if f.endswith(".pdbqt"):
        print(f"{f}")

In [ ]:
docked_dir = r"./data/docked"
print(f"docked/ 目录下共有 {len(os.listdir(docked_dir))} 个文件")
print("文件列表：")
for f in os.listdir(docked_dir):
    if f.endswith(".pdbqt"):
        print(f"  {f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 结合能数据（填你实际的数值）
data = {
    "Receptor": ["OR1A1", "OR5D13", "OR10J1", "OR4F3", "OR1D2"],
    "Citronellal": [-6.439, -5.363, -6.354, -5.862, -5.899],
    "Eugenol": [-6.772, -5.824, -6.943, -5.972, -5.924],
    "Linalool": [-6.301, -5.453, -6.128, -5.729, -5.901],
    "Limonene": [-6.924, -5.690, -6.841, -5.779, -6.236],
}

df = pd.DataFrame(data)
df

In [ ]:
# 把配体列取出来，受体列作为 X 轴
receptors = df["Receptor"]
ligands = ["Citronellal", "Eugenol", "Linalool", "Limonene"]

plt.figure(figsize=(10, 6))

for ligand in ligands:
    plt.plot(receptors, df[ligand], marker='o', label=ligand)

plt.xlabel("Receptor")
plt.ylabel("-ΔE Binding (kcal/mol)")
plt.title("OR Receptor Fingerprint Profiles")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
#以下为PDF和PNG格式的保存
plt.savefig("./figures/receptor_fingerprint.png", dpi=300, bbox_inches="tight")
plt.savefig("./figures/receptor_fingerprint.pdf", bbox_inches="tight")
plt.show()